In [0]:
from pyspark.sql.functions import current_timestamp , lit 
from pyspark.sql.types import StructType, StructField, StringType

dbutils.widgets.text("pipeline_name", "")
dbutils.widgets.text("activity_name", "")
dbutils.widgets.text("status", "")
dbutils.widgets.text("error_message", "")
dbutils.widgets.text("run_id", "")

pipeline_name = dbutils.widgets.get("pipeline_name")
activity_name = dbutils.widgets.get("activity_name")
status = dbutils.widgets.get("status")
error_message = dbutils.widgets.get("error_message")
run_id = dbutils.widgets.get("run_id")

schema = StructType([
    StructField("pipeline_name", StringType(), True),
    StructField("activity_name", StringType(), True),
    StructField("status", StringType(), True),
    StructField("error_message", StringType(), True),
    StructField("run_id", StringType(), True)
])

data = [(pipeline_name, activity_name, status, error_message, run_id)]

df_log = spark.createDataFrame(data, schema) \
    .withColumn("log_timestamp", current_timestamp()) \
    .withColumn("log_type", lit("ERROR"))

storage_account = "stbankamldev"

storage_key = dbutils.secrets.get(
    scope="aml-scope",
    key="storage-access-key"
)

spark.conf.set(
    f"fs.azure.account.key.{storage_account}.blob.core.windows.net",
    storage_key
)

df_log.write.mode("append").parquet(
    "wasbs://logs@stbankamldev.blob.core.windows.net/error_logs"
)

df_log.show(truncate=False)

In [0]:
df_log = spark.read.parquet(
    "wasbs://logs@stbankamldev.blob.core.windows.net/error_logs"
)

display(df_log)

In [0]:
df_pipeline = spark.read.parquet(
    "wasbs://logs@stbankamldev.blob.core.windows.net/pipeline_logs"
)

display(df_pipeline)

In [0]:
df_pipeline.printSchema()

In [0]:
df_error = spark.read.parquet(
    "wasbs://logs@stbankamldev.blob.core.windows.net/error_logs"
)

display(df_error.orderBy(df_error.log_timestamp.desc()))

In [0]:
storage_account = "stbankamldev"

storage_key = dbutils.secrets.get(
    scope="aml-scope",
    key="storage-access-key"
)

spark.conf.set(
    f"fs.azure.account.key.{storage_account}.blob.core.windows.net",
    storage_key
)

# error log fixed file 
files = dbutils.fs.ls("wasbs://logs@stbankamldev.blob.core.windows.net/error_logs_csv")

part_file = [f.path for f in files if f.name.startswith("part-")][0]

dbutils.fs.cp(part_file, 
              "wasbs://logs@stbankamldev.blob.core.windows.net/error_logs_fixed/error_logs.csv"
              )
              